# ⚙️ 05 Systematic Hyperparameter Tuning
Automate the search for optimal training configurations.

### 🎯 Goals:
1. Implement Grid Search over LR, Batch Size, and Dropout
2. Record real training metrics
3. Log all results to Drive

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive'): drive.mount('/content/drive')

import tensorflow as tf
import pandas as pd
import itertools
from tensorflow.keras import layers, models, applications

BASE_PATH = '/content/drive/MyDrive/DL/'
LOG_PATH = os.path.join(BASE_PATH, 'logs/')
if not os.path.exists(LOG_PATH): os.makedirs(LOG_PATH)

def train_and_evaluate(lr, bs, dr, opt_name):
    # Build MobileNetV2 for fast tuning
    model = models.Sequential([
        layers.Rescaling(1./127.5, offset=-1, input_shape=(224, 224, 3)),
        applications.MobileNetV2(include_top=False, weights='imagenet'),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(dr),
        layers.Dense(5, activation='softmax')
    ])
    model.layers[1].trainable = False
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr) if opt_name == 'adam' else tf.keras.optimizers.experimental.AdamW(learning_rate=lr)
    
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    
    # Note: Assumes train_ds and val_ds are in memory from Notebook 03
    history = model.fit(train_ds, epochs=3, validation_data=val_ds, batch_size=bs, verbose=0)
    
    val_acc = max(history.history['val_accuracy'])
    return val_acc

grid = {
    'lr': [1e-3, 1e-4],
    'bs': [16, 32],
    'dr': [0.2, 0.4],
    'opt': ['adam', 'adamw']
}

results = []
for lr, bs, dr, opt in itertools.product(*grid.values()):
    print(f"Testing: LR={lr}, BS={bs}, DR={dr}, OPT={opt}")
    acc = train_and_evaluate(lr, bs, dr, opt)
    results.append({'lr': lr, 'bs': bs, 'dr': dr, 'opt': opt, 'val_accuracy': acc})

df = pd.DataFrame(results)
df.to_csv(os.path.join(LOG_PATH, 'hyperparameter_results.csv'), index=False)
print("✅ Tuning complete.")